# Find, download, and process a NISAR GUNW with `nisar_pytools` and SnowIn

This notebook uses [`nisar_pytools`](https://github.com/ZachHoppinen/nisar_pytools) to search ASF's NISAR catalog and download one GUNW, then uses SnowIn to normalize its phase, derive local incidence, and calculate pairwise dSWE with the Leinss method.

The example searches for **GUNW** products because SnowIn 0.1 reads GUNW directly. GSLC products are single-date complex images; they must first be paired and processed into an interferogram with phase and geometry before SnowIn's phase-to-dSWE calculation can be used. SnowIn 0.1 does not read GSLC files directly.

## Environment and data access

Run this notebook from the SnowIn repository root. Install SnowIn's GUNW dependencies and `nisar_pytools` in the active Python 3.12 environment. Downloading from ASF may require an Earthdata login; follow the [`nisar_pytools` setup instructions](https://github.com/ZachHoppinen/nisar_pytools#installation). The search and download cells access external services and can download large files.

In [ ]:
%pip install -e ".[gunw,dev]"
%pip install nisar-pytools

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from nisar_pytools import download_urls, find_nisar

from snowin import compute_leinss_dswe
from snowin.io import open_gunw

AOI = [-115, 43, -114, 44]  # xmin, ymin, xmax, ymax; WGS84
START_DATE = "2025-07-01"
END_DATE = "2026-09-01"
TRACK = 77
DIRECTION = "ASCENDING"
DATA_DIR = Path("data/nisar_gunw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Search for GUNW products

The date range and track are example search settings. Adjust them for the area and season of interest. Inspect the returned rows and select a product before downloading; a GUNW can be several gigabytes.

In [ ]:
gunw_results = find_nisar(
    aoi=AOI,
    start_date=START_DATE,
    end_date=END_DATE,
    product_type="GUNW",
    path_number=TRACK,
    direction=DIRECTION,
)

print(f"Found {len(gunw_results)} GUNW products")
display(gunw_results.head())

In [ ]:
# Download one candidate by default. Review the search results above before
# changing PRODUCT_INDEX or downloading a larger time series.
if gunw_results.empty:
    raise RuntimeError(
        "No GUNW products matched. Change the AOI, dates, track, or direction."
    )

PRODUCT_INDEX = 0
selected = gunw_results.iloc[[PRODUCT_INDEX]]
display(selected)

downloaded_paths = download_urls(selected, str(DATA_DIR))
if not downloaded_paths:
    raise RuntimeError("nisar_pytools did not return a downloaded file path")

gunw_path = Path(downloaded_paths[0])
print(gunw_path)

## Inspect the downloaded product

`nisar_pytools` can open the HDF5 structure and expose the unwrapped phase and companion layers as labeled arrays. This small crop is for inspection; the SnowIn reader below opens the product and applies its own phase and metadata checks.

In [ ]:
from nisar_pytools import open_nisar
from nisar_pytools.utils.metadata import get_gunw

tree = open_nisar(str(gunw_path))
raw_gunw = get_gunw(tree)
raw_crop = raw_gunw[["unwrappedPhase", "coherenceMagnitude"]].isel(
    y=slice(0, 512), x=slice(0, 512)
).compute()
display(raw_crop)
tree.close()

## Open with SnowIn and calculate pairwise dSWE

`open_gunw` reads the product's phase convention and wavelength metadata and, by default, derives local incidence from the GUNW line-of-sight geometry and public Copernicus GLO-30 elevation data. The public DEM heights use the EGM2008 geoid while GUNW heights use the WGS84 ellipsoid. Without a supplied geoid-undulation correction, SnowIn marks the geometry result provisional; inspect that status before interpreting dSWE. Geometry calculation is eager, while the phase array can remain Dask-backed.

In [ ]:
pair = open_gunw(gunw_path)
print(pair)
print("Wavelength (m):", pair.attrs["wavelength_m"])
print(
    "Vertical datum status:",
    pair["incidence_angle"].attrs.get("vertical_datum_status", "not recorded"),
)

In [ ]:
dswe = compute_leinss_dswe(
    pair["phase"],
    pair["incidence_angle"],
    wavelength_m=pair.attrs["wavelength_m"],
)
dswe.name = "dswe_leinss"
dswe

In [ ]:
# Plot a small subset to keep interactive rendering responsive.
window = {"y": slice(0, 1000), "x": slice(0, 1000)}
phase_view = pair["phase"].isel(**window).compute()
dswe_view = dswe.isel(**window).compute()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
phase_view.plot(ax=axes[0], cmap="twilight", robust=True)
axes[0].set_title("Normalized GUNW phase (rad)")
dswe_view.plot(ax=axes[1], cmap="RdBu_r", robust=True)
axes[1].set_title("Pairwise dSWE, Leinss method (m water equivalent)")
plt.show()

## Interpret and extend

This result is **pairwise dSWE** (`SWE_secondary - SWE_reference`), not absolute SWE. No universal quality mask is applied here: inspect the GUNW coherence, connected-component, and support layers and choose a mask for the analysis. Reference-phase correction and time-series accumulation are separate SnowIn operations and require explicit inputs and a correctly ordered set of temporal edges.

For GSLC inputs, `nisar_pytools` also documents `stack_gslcs`, interferogram formation, coherence estimation, phase unwrapping, and local-incidence tools. That route must establish the GSLC pair's phase sign, units, grid alignment, and incidence-angle reference before passing arrays to SnowIn. See the [`nisar_pytools` processing guide](https://github.com/ZachHoppinen/nisar_pytools/blob/master/docs/processing.md) and SnowIn's [scientific conventions](../docs/scientific_conventions.md).